In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
runways = pd.read_csv(
    "../../data/processed/airports/runways/runways_feature_engineered.csv"
)

runways.head()

,id,airport_ref,airport_ident,length_ft,width_ft,surface,lighted,closed,le_ident,le_latitude_deg,le_longitude_deg,le_elevation_ft,le_heading_degT,le_displaced_threshold_ft,he_ident,he_latitude_deg,he_longitude_deg,he_elevation_ft,he_heading_degT,he_displaced_threshold_ft,surface_category,runway_length_category,runway_width_category,night_operation,operational_status,runway_capability,runway_risk_level
0,269408,6523,00A,80.0,80.0,ASPH-G,1,0,H1,NaN,NaN,NaN,NaN,NaN,UNKNOWN,NaN,NaN,NaN,NaN,NaN,ASPHALT,Very Short,Standard,Yes,Operational,Basic,Low
1,255155,6524,00AK,2500.0,40.0,GRVL,0,0,N,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,NaN,NaN,NaN,GRAVEL,Short,Narrow,No,Operational,Basic,Medium
2,254165,6525,00AL,2100.0,90.0,TURF,0,0,01,NaN,NaN,NaN,NaN,NaN,19,NaN,NaN,NaN,NaN,NaN,GRASS/TURF,Very Short,Standard,No,Operational,Basic,Medium
3,506792,506791,00AN,4517.0,60.0,GVL,0,0,3,NaN,NaN,NaN,NaN,NaN,21,NaN,NaN,NaN,NaN,NaN,OTHER,Short,Standard,No,Operational,Basic,Medium
4,322128,322127,00AS,1450.0,60.0,TURF,0,0,1,NaN,NaN,NaN,NaN,NaN,19,NaN,NaN,NaN,NaN,NaN,GRASS/TURF,Very Short,Standard,No,Operational,Basic,Medium


In [4]:
runways.shape

(48074, 27)

In [5]:
runways.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48074 entries, 0 to 48073
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         48074 non-null  int64  
 1   airport_ref                48074 non-null  int64  
 2   airport_ident              48074 non-null  object 
 3   length_ft                  48074 non-null  float64
 4   width_ft                   48074 non-null  float64
 5   surface                    48074 non-null  object 
 6   lighted                    48074 non-null  int64  
 7   closed                     48074 non-null  int64  
 8   le_ident                   48074 non-null  object 
 9   le_latitude_deg            15663 non-null  float64
 10  le_longitude_deg           15650 non-null  float64
 11  le_elevation_ft            13300 non-null  float64
 12  le_heading_degT            15075 non-null  float64
 13  le_displaced_threshold_ft  3067 non-null   flo

In [6]:
runway_count = (
    runways
    .groupby("airport_ident")
    .size()
    .reset_index(name="runway_count")
)

runway_count.head()

,airport_ident,runway_count
0,00A,1
1,00AK,1
2,00AL,1
3,00AN,1
4,00AS,1


In [7]:
longest_runway = (
    runways
    .groupby("airport_ident")["length_ft"]
    .max()
    .reset_index(name="longest_runway_ft")
)

longest_runway.head()

,airport_ident,longest_runway_ft
0,00A,80.0
1,00AK,2500.0
2,00AL,2100.0
3,00AN,4517.0
4,00AS,1450.0


In [8]:
average_length = (
    runways
    .groupby("airport_ident")["length_ft"]
    .mean()
    .round(1)
    .reset_index(name="average_runway_length_ft")
)

average_length.head()

,airport_ident,average_runway_length_ft
0,00A,80.0
1,00AK,2500.0
2,00AL,2100.0
3,00AN,4517.0
4,00AS,1450.0


In [9]:
average_width = (
    runways
    .groupby("airport_ident")["width_ft"]
    .mean()
    .round(1)
    .reset_index(name="average_runway_width_ft")
)

average_width.head()

,airport_ident,average_runway_width_ft
0,00A,80.0
1,00AK,40.0
2,00AL,90.0
3,00AN,60.0
4,00AS,60.0


In [10]:
lighted = (
    runways
    .groupby("airport_ident")["lighted"]
    .sum()
    .reset_index(name="lighted_runways")
)

lighted.head()

,airport_ident,lighted_runways
0,00A,1
1,00AK,0
2,00AL,0
3,00AN,0
4,00AS,0


In [11]:
closed = (
    runways
    .groupby("airport_ident")["closed"]
    .sum()
    .reset_index(name="closed_runways")
)

closed.head()

,airport_ident,closed_runways
0,00A,0
1,00AK,0
2,00AL,0
3,00AN,0
4,00AS,0


In [12]:
dominant_surface = (
    runways
    .groupby("airport_ident")["surface_category"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "UNKNOWN")
    .reset_index(name="dominant_surface")
)

dominant_surface.head()

,airport_ident,dominant_surface
0,00A,ASPHALT
1,00AK,GRAVEL
2,00AL,GRASS/TURF
3,00AN,OTHER
4,00AS,GRASS/TURF


In [13]:
capability_rank = {
    "Basic": 1,
    "Medium": 2,
    "High": 3
}

runways["capability_score"] = (
    runways["runway_capability"]
    .map(capability_rank)
)

In [14]:
maximum_capability = (
    runways
    .groupby("airport_ident")["capability_score"]
    .max()
    .reset_index()
)

In [15]:
reverse_rank = {
    1: "Basic",
    2: "Medium",
    3: "High"
}

maximum_capability["maximum_runway_capability"] = (
    maximum_capability["capability_score"]
    .map(reverse_rank)
)

maximum_capability.drop(
    columns="capability_score",
    inplace=True
)

maximum_capability.head()

,airport_ident,maximum_runway_capability
0,00A,Basic
1,00AK,Basic
2,00AL,Basic
3,00AN,Basic
4,00AS,Basic


In [16]:
airport_runway_summary = runway_count.merge(
    longest_runway,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    average_length,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    average_width,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    lighted,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    closed,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    dominant_surface,
    on="airport_ident"
)

airport_runway_summary = airport_runway_summary.merge(
    maximum_capability,
    on="airport_ident"
)

In [17]:
airport_runway_summary.head()

,airport_ident,runway_count,longest_runway_ft,average_runway_length_ft,average_runway_width_ft,lighted_runways,closed_runways,dominant_surface,maximum_runway_capability
0,00A,1,80.0,80.0,80.0,1,0,ASPHALT,Basic
1,00AK,1,2500.0,2500.0,40.0,0,0,GRAVEL,Basic
2,00AL,1,2100.0,2100.0,90.0,0,0,GRASS/TURF,Basic
3,00AN,1,4517.0,4517.0,60.0,0,0,OTHER,Basic
4,00AS,1,1450.0,1450.0,60.0,0,0,GRASS/TURF,Basic


In [18]:
airport_runway_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40979 entries, 0 to 40978
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   airport_ident              40979 non-null  object 
 1   runway_count               40979 non-null  int64  
 2   longest_runway_ft          40979 non-null  float64
 3   average_runway_length_ft   40979 non-null  float64
 4   average_runway_width_ft    40979 non-null  float64
 5   lighted_runways            40979 non-null  int64  
 6   closed_runways             40979 non-null  int64  
 7   dominant_surface           40979 non-null  object 
 8   maximum_runway_capability  40979 non-null  object 
dtypes: float64(3), int64(3), object(3)
memory usage: 2.8+ MB


In [19]:
airport_runway_summary.shape

(40979, 9)

In [20]:
print("Duplicate Airports:",
      airport_runway_summary["airport_ident"].duplicated().sum())

Duplicate Airports: 0


In [21]:
airport_runway_summary.to_csv(
    "../../data/processed/airports/airport_runway_summary.csv",
    index=False
)

print("Airport runway summary saved successfully.")

Airport runway summary saved successfully.


## Runway Aggregation Summary

The runway-level dataset was aggregated to create airport-level operational metrics. Since airports may contain multiple runways, aggregation was performed using the airport identifier. Features such as runway count, longest runway, average runway length, average runway width, number of lighted runways, number of closed runways, dominant runway surface, and maximum runway capability were derived. The resulting dataset contains one record per airport and provides a concise representation of airport runway infrastructure. This dataset will be integrated with airport, country, and region information to construct the Airport Master Dataset.